# Portfolio analytics

This notebook only orchestrates: it calls `transactions` (trade logic), `prices` (Yahoo Finance fetch + cache), `returns` (CAGR / HYSA benchmark math), and `visualization` (all charts). No logic lives in this notebook itself — see `docs/architecture.md` for the module map.

In [ ]:
from datetime import date
import polars as pl

from trades import returns, transactions, visualization
from trades.brokers.ibkr import main
from trades.config import AppConfig
from trades.market_data import prices
from trades.utils.frames import collect_if_lazy

AS_OF_DATE = date.today()  # change this to price the portfolio as of any past date

# Every tunable parameter lives on one of these config objects (see
# docs/architecture.md#configuration) — nothing here is a hidden default.
config = AppConfig()

## 1. Load, enrich, and aggregate trades

Trades come from the local IBKR ledger cache (`data/brokers/ibkr/ledger.csv`, kept fresh by running `notebooks/ibkr_sync.ipynb`), standardized from the canonical ledger onto the older, narrower trade schema by `preprocessing.standardize_ibkr_trades` (only `BUY` events on a real symbol — see `docs/architecture.md`, "Canonical trade schema"). `enrich_trades` then adds `usd_per_share`, and `aggregate_same_day_trades` merges same-day, same-symbol fills executed within 0.01% of each other into one row (summed shares/USD, recomputed $/share) — this is the dataset used for everything downstream.

In [ ]:
ledger = main.load_ledger(config)
raw = transactions.standardize_ibkr_trades(ledger, config)
enriched = transactions.enrich_trades(raw)
trades = collect_if_lazy(transactions.aggregate_same_day_trades(enriched, config))
print(f"{len(collect_if_lazy(raw))} standardized buys -> {len(trades)} aggregated trades")
trades

## 2. Investment schedule

Totals per symbol, invested-per-month (overall and per symbol), the daily investment timeline (with gaps between buys), and a pie breakdown with a menu to switch between whole-portfolio-by-symbol and any one symbol's by-date split.

In [ ]:
total_by_symbol = transactions.total_invested_by_symbol(trades)
print("Total invested to date, by symbol:")
print(total_by_symbol)
print(f"\nTotal invested to date, whole portfolio: ${total_by_symbol.sum():,.2f}")

In [ ]:
monthly = transactions.monthly_invested(trades)
monthly

In [ ]:
visualization.plot_monthly_invested(monthly).show()

In [ ]:
daily = transactions.daily_investment_timeline(trades)
visualization.plot_daily_investment_timeline(daily).show()

In [ ]:
pie_options = transactions.pie_chart_options(trades)
visualization.plot_investment_pie(pie_options).show()

## 3. Price history

One local cache file per symbol (`data/prices/{SYMBOL}.csv`), each call only fetching the date range missing since the last run — see `docs/architecture.md` for the cache design. History goes back to the first trade date across the whole portfolio.

In [ ]:
symbols = trades["symbol"].unique().sort().to_list()
first_trade_date = trades["trade_date"].min()

price_histories = prices.update_price_caches(
    symbols, since=first_trade_date, as_of=AS_OF_DATE, config=config
)
for symbol, history in price_histories.items():
    latest_close = history["close"][-1]
    print(f"{symbol}: {len(history)} trading days cached, latest close {latest_close:.2f}")

## 4. Returns vs. a HYSA benchmark

For each trade: current price, days held, total return, CAGR-style annualized return, the compounded HYSA return over the same window (rate set by `returns_config.hysa_annual_rate`, default 4%), and the resulting alpha. See `docs/returns.md` for the derivation of each step.

In [ ]:
def price_lookup(symbol: str, as_of: date) -> float | None:
    return prices.price_as_of(price_histories[symbol], as_of)


returns_df = returns.build_returns_table(
    trades, price_lookup, as_of=AS_OF_DATE, config=config
)

display_table = (
    returns_df
    .select([
        "trade_date",
        "symbol",
        "usd_per_share",
        "current_price",
        "days_held",
        "total_return_pct",
        "annualized_return_pct",
        "hysa_period_return_pct",
        "alpha_period_pct",
    ])
    .rename({"usd_per_share": "price_paid"})
)
display_table

In [ ]:

numeric_cols = [
    col
    for col, dtype in display_table.schema.items()
    if dtype in (pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
                 pl.Float32, pl.Float64)
]

display_rounded = display_table.with_columns([
    pl.col(c).round(2).alias(c)
    for c in numeric_cols
])

visualization.render_table(display_rounded, title=f"Returns as of {AS_OF_DATE}").show() # type: ignore

portfolio_alpha = returns.portfolio_alpha_pct(returns_df)
label = (
    f"Dollar-weighted portfolio alpha vs. {config.returns.hysa_annual_rate:.0%} HYSA "
    "(period, not annualized)"
)
print(f"{label}: {portfolio_alpha:+.2f}%")

## 5. Annualized return curve

Per-trade annualized return against days held, with a fitted trend and the flat HYSA benchmark line. Short holds annualize into large, noisy numbers by design — that's why the combined alpha above uses period alpha instead of this annualized figure.

In [ ]:
trend_x, trend_y = returns.fit_trend(
    returns_df.select("days_held").to_series().to_numpy(),
    returns_df.select("annualized_return_pct").to_series().to_numpy(),
    config,
)
visualization.plot_return_curve(returns_df, trend_x, trend_y, config).show()